# Lab Day 10: Finetune Qwen3-VL-4B (QLoRA) để định vị hư hại mặt đường
Notebook triển khai các bước trong GUIDE.md. Giữ nguyên các file đề bài và chỉ dùng số liệu sinh từ lần chạy thật.

**Colab setup:** clone or upload the complete repository into the runtime and run this notebook from the repository root; the notebook imports `lab_day10_solution.py` and calls the unchanged `scripts/` tools.

## 0. Môi trường (Colab T4)

In [3]:
from pathlib import Path
import os
import subprocess

repo = Path('/content/labrepo')
repo_url = 'https://github.com/5cuong/K4-Track-4-Day10-NguyenHoangCuong-2A202602473-VisionLanguageModel.git'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo)], check=True)
os.chdir(repo)
required = [Path('scripts/prepare_data.py'), Path('scripts/score.py'), Path('lab_day10_solution.py')]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError('Repository setup incomplete: ' + ', '.join(missing))
!nvidia-smi -L
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repo ready at', Path.cwd())

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-7e32cc89-d7f8-6e37-54cc-34f2e219239b)
Repo ready at /content/labrepo


In [4]:
import os,sys,zipfile,shutil,json,subprocess,importlib.metadata as metadata
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU unavailable; stop before any training.'
print('GPU:',torch.cuda.get_device_name(0),flush=True)
mydrive=Path('/content/drive/MyDrive')
recovery_path=mydrive/'day10_complete_recovery_20261015T035045Z.zip'
checkpoints_path=mydrive/'day10_checkpoints-20261015T012539Z-1-001.zip'
assert recovery_path.is_file() and checkpoints_path.is_file(), 'Required recovery archives missing.'
repo=Path('/content/day10_repo')
if repo.exists() and any(repo.iterdir()):
    raise FileExistsError(f'{repo} already contains files; refusing to overwrite it.')
repo.mkdir(parents=True,exist_ok=True)
root=mydrive/'day10_resume'
root.mkdir(parents=True,exist_ok=True)

def safe_target(base,name):
    rel=Path(name)
    if rel.is_absolute() or '..' in rel.parts: raise ValueError(f'Unsafe archive member: {name}')
    target=(base/rel).resolve(); target.relative_to(base.resolve()); return target

def extract_one(archive,name,base,relative_name=None):
    target=safe_target(base,relative_name or name)
    target.parent.mkdir(parents=True,exist_ok=True)
    with archive.open(name) as src, target.open('wb') as dst: shutil.copyfileobj(src,dst,1024*1024)
    return target

rz=zipfile.ZipFile(recovery_path)
manifest=json.loads(rz.read('day10_resume/manifest.json'))
needed=['day10_resume/workspace.zip','day10_resume/source.bundle','day10_resume/lab_day10_solution.py','day10_resume/day10_resume.py','day10_resume/requirements-resume.txt','day10_resume/manifest.json','day10_resume/REPORT.md','day10_resume/e1_legacy.json']
for name in needed:
    extract_one(rz,name,root,Path(name).relative_to('day10_resume').as_posix())
print('Recovery package files restored to Drive.',flush=True)
# Restore only the full-state E3 checkpoint that is needed to resume at step 20.
for name in rz.namelist():
    if name.startswith('day10_resume/runs/E3_r16/latest.json') or name.startswith('day10_resume/runs/E3_r16/rolling-1/'):
        rel=Path(name).relative_to('day10_resume').as_posix()
        if name.endswith('/') or name.endswith('.md'): continue
        extract_one(rz,name,root,rel)
print('E3 step-20 optimizer checkpoint restored.',flush=True)
# Recover the finished E1/E2 adapters, avoiding any repeat training.
cz=zipfile.ZipFile(checkpoints_path)
for key,prefix in [
 ('E1_r16','day10_checkpoints/e1-r16-20261014T144356Z/checkpoint-0300/'),
 ('E2_r16','day10_checkpoints/e2-r16-20261014T165944Z/checkpoint-0300/')]:
    cp=root/'runs'/key/'checkpoint-0300'
    for name in cz.namelist():
        if name.startswith(prefix) and not name.endswith('/') and not name.endswith('.md'):
            rel=name[len(prefix):]
            extract_one(cz,name,cp,rel)
    info=dict(manifest[key]); info.update({'setup':key[:2],'step':300,'steps':300,'adapter_dir':str(cp/'adapter'),'run_dir':str(cp.parent),'full_optimizer_state':False,'archived_adapter_only':True})
    (cp/'complete.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
    print(key,'step 300 adapter restored; no optimizer state in archive.',flush=True)
# Verify the included base-model shard index, then unpack it to ephemeral runtime storage.
base_local=Path('/content/day10_resume_base_model')
base_prefix='day10_resume/base_model/'
if not base_local.exists() or not (base_local/'model.safetensors.index.json').exists():
    base_local.mkdir(parents=True,exist_ok=True)
    members=[n for n in rz.namelist() if n.startswith(base_prefix) and not n.endswith('/') and not n.endswith('CACHE_READY.json')]
    for i,name in enumerate(members,1):
        rel=Path(name).relative_to(base_prefix).as_posix()
        extract_one(rz,name,base_local,rel)
        if i%4==0: print(f'Base-model files restored: {i}/{len(members)}',flush=True)
index=json.loads((base_local/'model.safetensors.index.json').read_text())
missing=[n for n in set(index['weight_map'].values()) if not (base_local/n).is_file()]
if missing: raise FileNotFoundError(f'Model shards missing: {missing}')
print('Base model cache complete:',len(set(index['weight_map'].values())),'shards.',flush=True)
# Safely restore the packed project workspace and exact recovery code in the runtime.
with zipfile.ZipFile(root/'workspace.zip') as wz:
    for info in wz.infolist(): safe_target(repo,info.filename)
    wz.extractall(repo)
shutil.copy2(root/'lab_day10_solution.py',repo/'lab_day10_solution.py')
shutil.copy2(root/'day10_resume.py',repo/'day10_resume.py')
(repo/'work'/'runs').mkdir(parents=True,exist_ok=True)
shutil.copy2(root/'manifest.json',repo/'work'/'runs'/'manifest.json')
# Update only generated run metadata to point to the restored E1/E2 adapters.
local_manifest=json.loads((repo/'work'/'runs'/'manifest.json').read_text())
for key in ('E1_r16','E2_r16'):
    p=root/'runs'/key/'checkpoint-0300'/'complete.json'
    local_manifest[key]=json.loads(p.read_text())
(repo/'work'/'runs'/'manifest.json').write_text(json.dumps(local_manifest,ensure_ascii=False,indent=2))
# Install only pinned lab dependencies that are not already at the recovery versions.
reqs=(root/'requirements-resume.txt').read_text().splitlines(); needed_pkgs=[]
for spec in reqs:
    if not spec.strip(): continue
    pkg,version=spec.split('==',1)
    try: ok=metadata.version(pkg)==version
    except metadata.PackageNotFoundError: ok=False
    if not ok: needed_pkgs.append(spec)
if needed_pkgs:
    print('Installing recovery-pinned dependencies:',needed_pkgs,flush=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',*needed_pkgs],check=True)
else: print('Pinned dependencies already match.',flush=True)
os.chdir(repo)
if str(repo) not in sys.path: sys.path.insert(0,str(repo))
import lab_day10_solution as sol, importlib
sol=importlib.reload(sol); sol.MODEL_ID=str(base_local)
import day10_resume as runner
runner=importlib.reload(runner); runner.ROOT=root; runner.REPO=repo
original_latest=runner.latest_checkpoint
def latest_checkpoint_with_archived_adapters(key):
    found=original_latest(key)
    if found or key not in ('E1_r16','E2_r16'): return found
    rows=[]
    for marker in (root/'runs'/key).glob('*/complete.json'):
        try:
            info=json.loads(marker.read_text()); cp=marker.parent
            if info.get('step',info.get('steps',0))>=300 and (cp/'adapter'/'adapter_model.safetensors').is_file():
                rows.append((info.get('step',info.get('steps',0)),info.get('saved_at',''),cp,info))
        except (OSError,ValueError,KeyError): pass
    return max(rows,key=lambda x:(x[0],x[1])) if rows else None
runner.latest_checkpoint=latest_checkpoint_with_archived_adapters
verified=sol.verify_dataset()
for key in ('E1_r16','E2_r16','E3_r16'):
    found=runner.latest_checkpoint(key)
    print('RESUME READY',key,'step',found[0] if found else None,'path',found[2] if found else None,flush=True)
print('DATASET',verified['meta']['dataset_version'],'| counts verified | workspace',repo,'| durable run root',root,flush=True)
print('TRAINER READY; no training was started in this cell.',flush=True)

GPU: NVIDIA A100-SXM4-40GB
Recovery package files restored to Drive.
E3 step-20 optimizer checkpoint restored.
E1_r16 step 300 adapter restored; no optimizer state in archive.
E2_r16 step 300 adapter restored; no optimizer state in archive.
Base-model files restored: 4/12
Base-model files restored: 8/12
Base-model files restored: 12/12
Base model cache complete: 2 shards.
Installing recovery-pinned dependencies: ['transformers==5.19.0', 'peft==0.21.2', 'bitsandbytes==0.50.2']
train: 1422 ảnh, boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
val: 276 ảnh, boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
test: 311 ảnh, boxes={'pothole': 121, 'crack': 299, 'manhole': 163}
dataset_version=d10-21e4070301; 18 sessions không giao nhau giữa split
RESUME READY E1_r16 step 300 path /content/drive/MyDrive/day10_resume/runs/E1_r16/checkpoint-0300
RESUME READY E2_r16 step 300 path /content/drive/MyDrive/day10_resume/runs/E2_r16/checkpoint-0300
RESUME READY E3_r16 step 300 path /content/dri

In [25]:
runner.run_pipeline()

Base cache pending: FileNotFoundError(2, 'No such file or directory')
SKIP COMPLETED TRAIN E3_r16
Archived E2 projector is incompatible with the current 4B base; retraining into runs/E2_r16_A100 to preserve the archive.


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

DRIVE CHECKPOINT SAVED E2_r16_A100 0 /content/drive/MyDrive/day10_resume/runs/E2_r16_A100/rolling-0 state_MB 0
TRAIN START E2_r16_A100 from 0 trainable 142135296 images 1422
E2_r16_A100 1/300 loss=1.75263 elapsed=0.1min ETA=43.6min VRAM=11.99GB
E2_r16_A100 10/300 loss=0.78041 elapsed=1.2min ETA=33.6min VRAM=13.27GB
E2_r16_A100 20/300 loss=0.70701 elapsed=2.3min ETA=31.8min VRAM=13.27GB
DRIVE CHECKPOINT SAVED E2_r16_A100 20 /content/drive/MyDrive/day10_resume/runs/E2_r16_A100/rolling-1 state_MB 1138
E2_r16_A100 30/300 loss=0.70064 elapsed=3.5min ETA=31.8min VRAM=13.27GB
E2_r16_A100 40/300 loss=0.61326 elapsed=4.6min ETA=30.2min VRAM=13.27GB
DRIVE CHECKPOINT SAVED E2_r16_A100 40 /content/drive/MyDrive/day10_resume/runs/E2_r16_A100/rolling-0 state_MB 1138
E2_r16_A100 50/300 loss=0.57789 elapsed=5.9min ETA=29.6min VRAM=13.28GB
E2_r16_A100 60/300 loss=0.62971 elapsed=7.0min ETA=28.2min VRAM=13.39GB
DRIVE CHECKPOINT SAVED E2_r16_A100 60 /content/drive/MyDrive/day10_resume/runs/E2_r16_A100/ro

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

PERSISTED PREDICTIONS E2 val 20 / 276
PERSISTED PREDICTIONS E2 val 40 / 276
PERSISTED PREDICTIONS E2 val 60 / 276
PERSISTED PREDICTIONS E2 val 80 / 276
PERSISTED PREDICTIONS E2 val 100 / 276
PERSISTED PREDICTIONS E2 val 120 / 276
PERSISTED PREDICTIONS E2 val 140 / 276
PERSISTED PREDICTIONS E2 val 160 / 276
PERSISTED PREDICTIONS E2 val 180 / 276
PERSISTED PREDICTIONS E2 val 200 / 276
PERSISTED PREDICTIONS E2 val 220 / 276
PERSISTED PREDICTIONS E2 val 240 / 276
PERSISTED PREDICTIONS E2 val 260 / 276
PERSISTED PREDICTIONS E2 val 276 / 276


...

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

PERSISTED PREDICTIONS E3 test 20 / 311
PERSISTED PREDICTIONS E3 test 40 / 311
PERSISTED PREDICTIONS E3 test 60 / 311
PERSISTED PREDICTIONS E3 test 80 / 311
PERSISTED PREDICTIONS E3 test 100 / 311
PERSISTED PREDICTIONS E3 test 120 / 311
PERSISTED PREDICTIONS E3 test 140 / 311
PERSISTED PREDICTIONS E3 test 160 / 311
PERSISTED PREDICTIONS E3 test 180 / 311
PERSISTED PREDICTIONS E3 test 200 / 311
PERSISTED PREDICTIONS E3 test 220 / 311
PERSISTED PREDICTIONS E3 test 240 / 311
PERSISTED PREDICTIONS E3 test 260 / 311
PERSISTED PREDICTIONS E3 test 280 / 311
PERSISTED PREDICTIONS E3 test 300 / 311
PERSISTED PREDICTIONS E3 test 311 / 311


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

DRIVE CHECKPOINT SAVED E4_data_size_0.1 0 /content/drive/MyDrive/day10_resume/runs/E4_data_size_0.1/rolling-0 state_MB 0
TRAIN START E4_data_size_0.1 from 0 trainable 148426752 images 142
E4_data_size_0.1 1/300 loss=2.21487 elapsed=0.2min ETA=56.2min VRAM=12.10GB
E4_data_size_0.1 10/300 loss=0.74603 elapsed=1.5min ETA=43.1min VRAM=13.41GB
SAFE TO RESUME FROM DRIVE: 0


KeyboardInterrupt: 

In [3]:
import sys, json
from pathlib import Path
print("GPU", torch.cuda.get_device_name(0), "cuda", torch.cuda.is_available())
print("runner_root", getattr(runner, "ROOT", None), "runner_type", type(runner).__name__)
print("last_exception", repr(getattr(sys, "last_value", None)))
print("last_checkpoint_results:")
for k in ("E1_r16", "E2_r16", "E3_r16", "E4_r16"):
    try:
        print(k, runner.latest_checkpoint(k))
    except Exception as e:
        print(k, type(e).__name__, str(e)[:500])
root = Path(getattr(runner, "ROOT", "/content/drive/MyDrive/day10_resume"))
print("error_records:")
for pattern in ("last_error.json", "error.json"):
    for p in root.rglob(pattern):
        print(p, p.read_text(encoding="utf-8", errors="replace")[:3000])
print("recent_run_files:")
for p in sorted((root / "runs").glob("*/"), key=lambda x: x.name):
    files = sorted([str(f.relative_to(root)) for f in p.rglob("*") if f.is_file() and (f.name.endswith(".json") or f.suffix in (".log", ".txt"))])
    print(p.name, files[-12:])

NameError: name 'torch' is not defined

In [43]:
import json; from pathlib import Path; mp=sol._manifest_path(); m=json.loads(mp.read_text(encoding='utf-8')); e1=dict(e1_run_info); e1.update({'setup':'E1','rank':16,'alpha':32,'steps':300,'step':300,'projector_saved':False,'full_optimizer_state':False}); e2=json.loads(Path('/content/drive/MyDrive/day10_resume/runs/E2_r16_A100/checkpoint-0300/complete.json').read_text(encoding='utf-8')); e3=json.loads(Path('/content/drive/MyDrive/day10_resume/runs/E3_r16/checkpoint-0300/complete.json').read_text(encoding='utf-8')); assert all(Path(e['adapter_dir']).is_dir() and (Path(e['adapter_dir'])/'adapter_model.safetensors').is_file() for e in (e1,e2,e3)); assert all((Path(e['adapter_dir'])/'projector.pt').is_file() for e in (e2,e3)); m.update({'E1_r16':e1,'E2_r16':e2,'E3_r16':e3}); tp=mp.with_suffix('.json.tmp'); tp.write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); tp.replace(mp); m2=sol._manifest(); print('MANIFEST_UPDATED',mp); print([(k,m2[k].get('step',m2[k].get('steps')),m2[k].get('trainable_params'),m2[k].get('peak_vram_gb'),m2[k].get('train_minutes'),m2[k].get('adapter_dir')) for k in ('E1_r16','E2_r16','E3_r16')]); print('E4_RETAINED',all(k in m2 for k in ('E3_r4','E3_r64','E4_rank_4','E4_rank_64')))

MANIFEST_UPDATED /content/day10_repo/work/runs/manifest.json
[('E1_r16', 300, 33030144, 29.2012, 11.2356, '/content/day10_repo/work/runs/e1-r16-20261016T015036Z/adapter'), ('E2_r16', 300, 142135296, 13.387965679168701, 36.13225569638332, '/content/drive/MyDrive/day10_resume/runs/E2_r16_A100/checkpoint-0300/adapter'), ('E3_r16', 300, 148426752, 10.132108688354492, 45.51973023421667, '/content/drive/MyDrive/day10_resume/runs/E3_r16/checkpoint-0300/adapter')]
E4_RETAINED True


In [21]:

import importlib, torch
sol=importlib.reload(sol)
manifest=sol._manifest()
print("best setup",manifest.get("best_setup"),"validation F1",manifest.get("validation_f1"))
clean_apply=sol.apply_setup
def apply_setup_no_gc(model, setup_name, rank=16, _apply=clean_apply):
    model.gradient_checkpointing_enable=lambda **kwargs: None
    if hasattr(model,"config"):
        model.config.use_cache=False
    return _apply(model,setup_name,rank=rank)
sol.apply_setup=apply_setup_no_gc
print("patched from clean module",sol.__file__,"GPU",torch.cuda.get_device_name(0))

best setup E3 validation F1 {'E1': 0.18258664412510567, 'E2': 0.28, 'E3': 0.3544520547945206}
patched from clean module /content/day10_repo/lab_day10_solution.py GPU NVIDIA A100-SXM4-40GB


In [24]:
results=sol.run_rank_ablation()
print("E4 RESULTS", results)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Inference: 50/311
Inference: 100/311
Inference: 150/311
Inference: 200/311
Inference: 250/311
Inference: 300/311
Inference: 311/311
Đã lưu 311 dự đoán: /content/day10_repo/work/e4_rank_4.jsonl


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

E3 r=64 step=1/300 loss=1.95685 lr=2.67e-05
E3 r=64 step=10/300 loss=0.73190 lr=0.000147
E3 r=64 step=20/300 loss=0.66130 lr=0.0002
E3 r=64 step=30/300 loss=0.56473 lr=0.000199
E3 r=64 step=40/300 loss=0.70027 lr=0.000196
E3 r=64 step=50/300 loss=0.65316 lr=0.000193
E3 r=64 step=60/300 loss=0.61416 lr=0.000188
E3 r=64 step=70/300 loss=0.63566 lr=0.000182
E3 r=64 step=80/300 loss=0.62108 lr=0.000175
E3 r=64 step=90/300 loss=0.66691 lr=0.000168
E3 r=64 step=100/300 loss=0.57083 lr=0.000159
Drive backup: /content/drive/MyDrive/day10_checkpoints/e3-r64-20261016T001918Z/checkpoint-0100
E3 r=64 step=110/300 loss=0.64468 lr=0.00015
E3 r=64 step=120/300 loss=0.55869 lr=0.00014
E3 r=64 step=130/300 loss=0.61979 lr=0.00013
E3 r=64 step=140/300 loss=0.57580 lr=0.000119
E3 r=64 step=150/300 loss=0.53263 lr=0.000108
E3 r=64 step=160/300 loss=0.63474 lr=9.72e-05
E3 r=64 step=170/300 loss=0.58119 lr=8.63e-05
E3 r=64 step=180/300 loss=0.55680 lr=7.55e-05
E3 r=64 step=190/300 loss=0.45731 lr=6.49e-05
E

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Inference: 50/311
Inference: 100/311
Inference: 150/311
Inference: 200/311
Inference: 250/311
Inference: 300/311
Inference: 311/311
Đã lưu 311 dự đoán: /content/day10_repo/work/e4_rank_64.jsonl
E4 CI95 rank 4=[0.36860670194003525, 0.4498594189315839]; rank 64=[0.3884372177055104, 0.46875]; overlap=True
E4 RESULTS {4: {'n_images': 311, 'parse_valid_rate': 1.0, 'missing_predictions': 0, 'micro': {'precision': 0.42201834862385323, 'recall': 0.39451114922813035, 'f1': 0.40780141843971635, 'tp': 230, 'fp': 315, 'fn': 353}, 'per_class': {'pothole': {'precision': 0.3220338983050847, 'recall': 0.3140495867768595, 'f1': 0.3179916317991632, 'tp': 38, 'fp': 80, 'fn': 83}, 'crack': {'precision': 0.31313131313131315, 'recall': 0.3110367892976589, 'f1': 0.31208053691275167, 'tp': 93, 'fp': 204, 'fn': 206}, 'manhole': {'precision': 0.7615384615384615, 'recall': 0.6073619631901841, 'f1': 0.6757679180887373, 'tp': 99, 'fp': 31, 'fn': 64}, 'other': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'tp': 0, '

In [37]:
import torch, json, lab_day10_solution as sol
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)
print('Starting fresh measured E1 run; prior Drive checkpoint is preserved.',flush=True)
e1_run_info = sol._train_one('E1',rank=16,steps=300)
print('E1_RUN_COMPLETE',json.dumps(e1_run_info,ensure_ascii=False),flush=True)


Starting fresh measured E1 run; prior Drive checkpoint is preserved.


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

E1 r=16 step=1/300 loss=1.92171 lr=2.67e-05
E1 r=16 step=10/300 loss=0.85786 lr=0.000147
E1 r=16 step=20/300 loss=0.77029 lr=0.0002
E1 r=16 step=30/300 loss=0.64286 lr=0.000199
E1 r=16 step=40/300 loss=0.61954 lr=0.000196
E1 r=16 step=50/300 loss=0.62137 lr=0.000193
E1 r=16 step=60/300 loss=0.63666 lr=0.000188
E1 r=16 step=70/300 loss=0.64457 lr=0.000182
E1 r=16 step=80/300 loss=0.59322 lr=0.000175
E1 r=16 step=90/300 loss=0.57869 lr=0.000168
E1 r=16 step=100/300 loss=0.66412 lr=0.000159
Drive backup: /content/drive/MyDrive/day10_checkpoints/e1-r16-20261016T015036Z/checkpoint-0100
E1 r=16 step=110/300 loss=0.54240 lr=0.00015
E1 r=16 step=120/300 loss=0.61577 lr=0.00014
E1 r=16 step=130/300 loss=0.54500 lr=0.00013
E1 r=16 step=140/300 loss=0.60536 lr=0.000119
E1 r=16 step=150/300 loss=0.63857 lr=0.000108
E1 r=16 step=160/300 loss=0.62534 lr=9.72e-05
E1 r=16 step=170/300 loss=0.61234 lr=8.63e-05
E1 r=16 step=180/300 loss=0.59697 lr=7.55e-05
E1 r=16 step=190/300 loss=0.62046 lr=6.49e-05
E

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')
if not kaggle_username or not kaggle_key:
    raise RuntimeError('Hãy thêm KAGGLE_USERNAME và KAGGLE_KEY trong Colab Secrets.')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key


E2 resume route ready; old E2 checkpoint preserved.


## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from lab_day10_solution import verify_dataset, visualize_ground_truth



checked = verify_dataset()  # version, exact split/class counts, and session isolation

visualize_ground_truth(checked['splits']['train'], count=6)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab_day10_solution import yolo_to_box, objects_to_answer

from scripts.labcommon import read_jsonl



example = read_jsonl('data/splits/train.jsonl')[0]

print('Grounding target:', objects_to_answer(example['objects']))

print('One converted box:', yolo_to_box(example['objects'][0]['label'], example['objects'][0]['yolo']) if example['objects'] else '[]')

## 3. E0: zero-shot

In [ ]:
from lab_day10_solution import run_zero_shot



# Generates work/e0.jsonl from all 311 official test images.

run_zero_shot()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --split test --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from lab_day10_solution import inspect_masking_and_collation



# Verifies assistant-only labels, JSON + <|im_end|>, image tensor concatenation,

# and reports the visual token count produced by this installed processor.

processor = inspect_masking_and_collation()

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from lab_day10_solution import apply_setup



# The concrete setup is applied by the training loop, then the injected module

# count is checked against 252 LLM targets (E1/E2) or 348 total (E3).

SETUP_DESCRIPTIONS = {

    'E1': 'LLM LoRA; ViT and Projector frozen',

    'E2': 'E1 + full float32 visual.merger and deepstack mergers',

    'E3': 'E2 + ViT LoRA on qkv / linear_fc1 / linear_fc2',

}

for name, description in SETUP_DESCRIPTIONS.items():

    print(name, '—', description)

## 6. Vòng huấn luyện

In [ ]:
from lab_day10_solution import train_supervised_sweep



# Trains E1, E2, E3 for 300 optimizer steps each; batch=2, accumulation=4.

# Selects the best setup using validation F1 only and backs up every 100 steps.

sft_runs = train_supervised_sweep()

## 7. Suy luận và chấm E1–E3

In [44]:
from lab_day10_solution import evaluate_supervised



# Reloads each adapter (including separately saved projector weights), evaluates

# the official 311-image test split, writes e1/e2/e3 evidence, and draws 6 examples.

test_scores = evaluate_supervised()

for name, score in test_scores.items():

    print(name, score['micro']['f1'], score['f1_ci95'], score['parse_valid_rate'])

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Inference: 50/311
Inference: 100/311
Inference: 150/311
Inference: 200/311
Inference: 250/311
Inference: 300/311
Inference: 311/311
Đã lưu 311 dự đoán: /content/day10_repo/work/e1.jsonl


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Inference: 50/311
Inference: 100/311
Inference: 150/311
Inference: 200/311
Inference: 250/311
Inference: 300/311
Inference: 311/311
Đã lưu 311 dự đoán: /content/day10_repo/work/e2.jsonl


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Inference: 50/311
Inference: 100/311
Inference: 150/311
Inference: 200/311
Inference: 250/311
Inference: 300/311
Inference: 311/311
Đã lưu 311 dự đoán: /content/day10_repo/work/e3.jsonl
Đã lưu 6 ảnh GT/Pred trên test: /content/day10_repo/report/figures/best_validation_setup_test_examples.png
Acceptance E1: parse_valid_rate=0.9904 (PASS; cần ≥0.95); F1 gain=0.1742 (PASS; cần ≥0.10)
E1 0.26291931097008164 [0.229020979020979, 0.30313588850174217] 0.9903536977491961
E2 0.34815515610217596 [0.3075489282385835, 0.388560157790927] 1.0
E3 0.425414364640884 [0.3832442067736186, 0.46796116504854374] 1.0


In [45]:
import subprocess as sp
for n in ('e1','e2','e3'):
    r=sp.run(['python','scripts/score.py','--pred',f'work/{n}.jsonl','--split','test','--name',n.upper(),'--out',f'report/evidence/{n}.json'],text=True,capture_output=True)
    print(r.stdout)
    if r.returncode:
        raise RuntimeError(r.stderr)
    print('SCORED',n)

E1 [test] n=311 valid=0.990 P=0.279 R=0.249 F1=0.263 CI95=0.229-0.303 meanIoU=0.672 FP/img=1.21
   pothole  P=0.237 R=0.149 F1=0.183
   crack    P=0.169 R=0.197 F1=0.182
   manhole  P=0.716 R=0.417 F1=0.527

SCORED e1
E2 [test] n=311 valid=1.000 P=0.388 R=0.316 F1=0.348 CI95=0.308-0.389 meanIoU=0.674 FP/img=0.93
   pothole  P=0.325 R=0.207 F1=0.253
   crack    P=0.276 R=0.258 F1=0.266
   manhole  P=0.695 R=0.503 F1=0.584

SCORED e2
E3 [test] n=311 valid=1.000 P=0.459 R=0.396 F1=0.425 CI95=0.383-0.468 meanIoU=0.696 FP/img=0.87
   pothole  P=0.291 R=0.264 F1=0.277
   crack    P=0.367 R=0.331 F1=0.348
   manhole  P=0.813 R=0.613 F1=0.699

SCORED e3


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [27]:
from lab_day10_solution import check_iou_reward_cases, run_rank_ablation, run_grpo



# E4 changes only LoRA rank from the best validation setup; alpha remains 2*rank.

e4_scores = run_rank_ablation()



# E5 is optional. The five deterministic reward cases can be checked without GPU training.

reward_cases = check_iou_reward_cases()

RUN_E5 = False  # Set True only to attempt the optional 60-step TRL GRPO run on T4.

if RUN_E5:

    e5_score = run_grpo(steps=60)



# Before submission, complete REPORT.md with your actual evidence and run:

# python scripts/check_submission.py


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 3894
Number of tensors saved during recomputation: 69.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


In [46]:
from pathlib import Path
import json, subprocess
root=Path('/content/day10_repo')
print('ROOT', root, 'exists=', root.is_dir(), 'cwd=', Path.cwd())
for name in ['README.md','GUIDE.md','RUBRIC.md']:
 p=root/name
 lines=p.read_text(encoding='utf-8',errors='replace').splitlines() if p.exists() else []
 hits=[(i+1,s) for i,s in enumerate(lines) if any(k in s.lower() for k in ['e0','e1','e2','e3','e4','check_submission','commit','gpu','nộp'])]
 print('\n---',name,'---')
 for i,s in hits[:45]: print(f'{i}: {s[:240]}')
for name in ['e0.json','e1.json','e2.json','e3.json']:
 p=root/'report/evidence'/name
 print('EVIDENCE',name,p.exists(),p.stat().st_size if p.exists() else None)
 if p.exists(): print(p.read_text(encoding='utf-8')[:1100])
for p in sorted((root/'report/evidence').glob('e4*.json')): print('E4_EVIDENCE',p.name,p.stat().st_size,p.read_text(encoding='utf-8')[:1100])
mp=root/'work/runs/manifest.json'
if mp.exists():
 m=json.loads(mp.read_text()); print('MANIFEST_KEYS',sorted(m.keys()))
 for k in ['E0','E1_r16','E2_r16','E3_r16','E3_r4','E3_r64','E4_rank_4','E4_rank_64','E4_data_size_0.1']:
  if k in m: print('MANIFEST',k,json.dumps(m[k],ensure_ascii=False)[:900])
print('\n--- REPORT.md ---')
print((root/'REPORT.md').read_text(encoding='utf-8',errors='replace')[:14000])
for args in [['git','status','--short'],['git','branch','--show-current'],['git','log','-5','--format=%s']]:
 r=subprocess.run(args,cwd=root,text=True,capture_output=True); print('GIT',args[1],r.stdout.strip(),r.stderr.strip()[:300])
import os, json, subprocess, re
from pathlib import Path
root=Path('/content/day10_repo')
os.chdir(root)
print('ROOT',root,'GPU',subprocess.run(['nvidia-smi','-L'],text=True,capture_output=True).stdout.strip())
for fn in ('README.md','GUIDE.md','RUBRIC.md'):
 p=root/fn; print('\n###',fn,'exists',p.exists())
 if p.exists():
  lines=p.read_text(encoding='utf-8',errors='replace').splitlines()
  for i,line in enumerate(lines):
   if re.search(r'E0|E1|E2|E3|E4|submit|submission|report|checkpoint|must|required|bắt buộc|yêu cầu|GPU',line,re.I):
    lo=max(0,i-1); hi=min(len(lines),i+2)
    print(f'[{i+1}] '+'\n'.join(lines[lo:hi]))
print('\n### REPORT.md')
p=root/'REPORT.md'; print(p.read_text(encoding='utf-8',errors='replace')[:18000] if p.exists() else 'MISSING')
print('\n### evidence')
for p in sorted((root/'report/evidence').glob('*.json')):
 try:
  d=json.loads(p.read_text()); print(p.name,json.dumps(d,ensure_ascii=False)[:1800])
 except Exception as e: print(p.name,'READ_ERROR',repr(e))
print('\n### prediction_files')
for n in ('e0','e1','e2','e3'):
 p=root/f'work/{n}.jsonl'; print(n,p.exists(),p.stat().st_size if p.exists() else None, sum(1 for _ in p.open()) if p.exists() else None)
print('\n### manifest entries')
mp=root/'work/runs/manifest.json'
if mp.exists():
 m=json.loads(mp.read_text()); print('keys',list(m));
 for k,v in m.items():
  if k.startswith(('E0','E1','E2','E3','E4')): print(k,json.dumps(v,ensure_ascii=False)[:1400])
else: print('MANIFEST MISSING',mp)
print('\n### run files')
for p in sorted((root/'work/runs').glob('*/')):
 fs=[str(x.relative_to(root)) for x in p.rglob('*') if x.is_file() and (x.suffix in ('.json','.log','.txt'))]
 print(p.name,fs[-16:])
print('\n### check_submission.py (read only)')
p=root/'scripts/check_submission.py'; print(p.read_text(encoding='utf-8',errors='replace')[:16000] if p.exists() else 'MISSING')
for args in (['git','status','--short'],['git','branch','--show-current'],['git','log','-5','--format=%s']):
import os, json, subprocess, re
from pathlib import Path
root=Path('/content/day10_repo')
os.chdir(root)
print('ROOT',root,'GPU',subprocess.run(['nvidia-smi','-L'],text=True,capture_output=True).stdout.strip())
for fn in ('README.md','GUIDE.md','RUBRIC.md'):
 p=root/fn; print('\n###',fn,'exists',p.exists())
 if p.exists():
  lines=p.read_text(encoding='utf-8',errors='replace').splitlines()
  for i,line in enumerate(lines):
   if re.search(r'E0|E1|E2|E3|E4|submit|submission|report|checkpoint|must|required|bắt buộc|yêu cầu|GPU',line,re.I):
    lo=max(0,i-1); hi=min(len(lines),i+2)
    print(f'[{i+1}] '+'\n'.join(lines[lo:hi]))
print('\n### REPORT.md')
p=root/'REPORT.md'; print(p.read_text(encoding='utf-8',errors='replace')[:18000] if p.exists() else 'MISSING')
print('\n### evidence')
for p in sorted((root/'report/evidence').glob('*.json')):
 try:
  d=json.loads(p.read_text()); print(p.name,json.dumps(d,ensure_ascii=False)[:1800])
 except Exception as e: print(p.name,'READ_ERROR',repr(e))
print('\n### prediction_files')
for n in ('e0','e1','e2','e3'):
 p=root/f'work/{n}.jsonl'; print(n,p.exists(),p.stat().st_size if p.exists() else None, sum(1 for _ in p.open()) if p.exists() else None)
print('\n### manifest entries')
mp=root/'work/runs/manifest.json'
if mp.exists():
 m=json.loads(mp.read_text()); print('keys',list(m));
 for k,v in m.items():
  if k.startswith(('E0','E1','E2','E3','E4')): print(k,json.dumps(v,ensure_ascii=False)[:1400])
else: print('MANIFEST MISSING',mp)
print('\n### run files')
for p in sorted((root/'work/runs').glob('*/')):
 fs=[str(x.relative_to(root)) for x in p.rglob('*') if x.is_file() and (x.suffix in ('.json','.log','.txt'))]
 print(p.name,fs[-16:])
print('\n### check_submission.py (read only)')
p=root/'scripts/check_submission.py'; print(p.read_text(encoding='utf-8',errors='replace')[:16000] if p.exists() else 'MISSING')
for args in (['git','status','--short'],['git','branch','--show-current'],['git','log','-5','--format=%s']):
 r=subprocess.run(args,cwd=root,text=True,capture_output=True); print('GIT',args[1],r.returncode,r.stdout.strip(),r.stderr.strip()[:400])

IndentationError: expected an indented block after 'for' statement on line 61 (2358287677.py, line 62)

In [47]:
import os,json,subprocess,re
from pathlib import Path
root=Path('/content/day10_repo'); os.chdir(root)
print('GPU',subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.strip())
for fn in ['README.md','GUIDE.md','RUBRIC.md']:
 lines=(root/fn).read_text(encoding='utf-8',errors='replace').splitlines(); print('\n###',fn)
 hits=[(i,l) for i,l in enumerate(lines) if re.search(r'E0|E1|E2|E3|E4|submit|submission|report|required|bắt buộc|yêu cầu',l,re.I)]
 for i,l in hits[:50]: print(f'{i+1}: {l[:260]}')
print('\n### REPORT.md'); print((root/'REPORT.md').read_text(encoding='utf-8',errors='replace')[:10000])
print('\n### evidence')
for p in sorted((root/'report/evidence').glob('*.json')):
 d=json.loads(p.read_text()); print(p.name,json.dumps(d,ensure_ascii=False)[:900])
print('\n### predictions')
for n in ['e0','e1','e2','e3']:
 p=root/f'work/{n}.jsonl'; print(n,p.exists(),p.stat().st_size if p.exists() else 0,sum(1 for _ in p.open()) if p.exists() else 0)
mp=root/'work/runs/manifest.json'; m=json.loads(mp.read_text()) if mp.exists() else {}; print('MANIFEST KEYS',list(m))
for k,v in m.items():
 if k.startswith(('E0','E1','E2','E3','E4')): print(k,json.dumps(v,ensure_ascii=False)[:1000])
print('CHECKER', (root/'scripts/check_submission.py').read_text(encoding='utf-8',errors='replace')[:12000])
for a in [['git','status','--short'],['git','branch','--show-current'],['git','log','-4','--format=%s']]:
 r=subprocess.run(a,cwd=root,capture_output=True,text=True); print('GIT',a[1],r.returncode,r.stdout.strip(),r.stderr.strip()[:250])

GPU GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-23190c6d-e0ec-04c0-86d8-858cd9549b8a)

### README.md
11: | [`REPORT.md`](REPORT.md) | Mẫu báo cáo 10 câu |
13: | `configs/e0…e5.yaml` | Siêu tham số từng thí nghiệm |
16: | `scripts/check_submission.py` | Kiểm tra bài trước khi nộp |
63: | Mã | Setup | Bắt buộc |
65: | E0 | Zero-shot, không huấn luyện | ✔ |
66: | E1 | LoRA chỉ LLM; ViT và projector đóng băng | ✔ |
67: | E2 | E1 + mở đầy đủ projector (`visual.merger`, `visual.deepstack_merger_list`, ≈ 109M tham số) | ✔ |
68: | E3 | E2 + LoRA trên các khối attention/MLP của ViT | ✔ |
69: | E4 | Ablation: chọn **một** biến (rank / kích thước dữ liệu / thang toạ độ) | ✔ (chọn 1) |
82: ## 6. Yêu cầu và cách nộp
85: - Nộp **repo GitHub công khai** (≥ 5 commit) gồm: `report/evidence/e0–e3.json`, ít nhất một `e4*.json`, `REPORT.md` đã điền, ảnh minh hoạ, notebook/mã của bạn.
86: - Trước khi nộp: `python scripts/check_submission.py` không còn FAIL.

### GUIDE.md
3: > Đọc [`README.md`](README.md) trước

In [48]:
import json, subprocess
from pathlib import Path
root=Path('/content/day10_repo')
for f in ['/content/day10_repo/work/e0.jsonl','/content/drive/MyDrive/day10_resume/work/e0.jsonl','/content/drive/MyDrive/day10_resume/e0.jsonl']:
 p=Path(f); print('E0FILE',f,p.exists(),p.stat().st_size if p.exists() else 0)
for p in sorted(root.glob('**/e0*.jsonl'))[:20]: print('E0FOUND',p,p.stat().st_size)
for k in ['e0','e1','e2','e3','e4_rank_4','e4_rank_64']:
 p=root/'report/evidence'/f'{k}.json'
 if p.exists():
  d=json.loads(p.read_text()); print('SCORE',k,'n',d.get('n_images'),'valid',d.get('parse_valid_rate'),'F1',d.get('micro',{}).get('f1'),'CI',d.get('f1_ci95'),'FP/img',d.get('fp_per_image'),'classes',{n:v.get('f1') for n,v in d.get('per_class',{}).items() if n!='other'},'meta',d.get('meta'),'split',d.get('split'),'ver',d.get('dataset_version'))
mp=root/'data/splits/meta.json'; print('DATA_META',mp.exists(),mp.read_text()[:8000] if mp.exists() else '')
print('TEST_FILES',[str(p.relative_to(root)) for p in (root/'tests').glob('*') if p.is_file()])
for folder in ['/content/labrepo','/content']:
 p=Path(folder); print('REPO',folder,(p/'.git').exists())
 if (p/'.git').exists():
  for a in [['git','-C',folder,'status','--short'],['git','-C',folder,'branch','--show-current'],['git','-C',folder,'rev-list','--count','HEAD'],['git','-C',folder,'log','-8','--format=%h %s'],['git','-C',folder,'remote','get-url','origin']]:
   r=subprocess.run(a,text=True,capture_output=True); print('GIT',a[3:],r.returncode,r.stdout.strip(),r.stderr.strip()[:250])
for f in ['/content/day10_resume_base_model/model.safetensors.index.json','/content/day10_resume_base_model/config.json']:
 p=Path(f); print('MODELFILE',f,p.exists())
 if p.exists() and p.name=='model.safetensors.index.json':
  d=json.loads(p.read_text()); print('SHARDS',sorted(set(d['weight_map'].values())),'KEYS',list(d['weight_map'])[:24])
for f in sorted((root/'report/figures').glob('*')): print('FIGURE',f.name,f.stat().st_size)


E0FILE /content/day10_repo/work/e0.jsonl False 0
E0FILE /content/drive/MyDrive/day10_resume/work/e0.jsonl False 0
E0FILE /content/drive/MyDrive/day10_resume/e0.jsonl False 0
SCORE e0 n 311 valid 0.8327974276527331 F1 0.08869179600886919 CI [0.06299212598425197, 0.11555555555555556] FP/img 0.8971061093247589 classes {'pothole': 0.10884353741496598, 'crack': 0.032128514056224904, 'manhole': 0.18677042801556423} meta {} split test ver d10-21e4070301
SCORE e1 n 311 valid 0.9903536977491961 F1 0.26291931097008164 CI [0.229020979020979, 0.30313588850174217] FP/img 1.2057877813504823 classes {'pothole': 0.182741116751269, 'crack': 0.1820987654320988, 'manhole': 0.5271317829457364} meta {} split test ver d10-21e4070301
SCORE e2 n 311 valid 1.0 F1 0.34815515610217596 CI [0.3075489282385835, 0.388560157790927] FP/img 0.932475884244373 classes {'pothole': 0.25252525252525254, 'crack': 0.2664359861591695, 'manhole': 0.5836298932384341} meta {} split test ver d10-21e4070301
SCORE e3 n 311 valid 1.0

In [49]:
from pathlib import Path
import zipfile, json, re
root=Path('/content/day10_repo')
archive=Path('/content/drive/MyDrive/day10_complete_recovery_20261015T035045Z.zip')
with zipfile.ZipFile(archive) as z:
    hits=[n for n in z.namelist() if re.search(r'e0.*jsonl|jsonl.*e0', n, re.I)]
    print('E0_ARCHIVE_HITS', hits[:30])
    for n in hits[:5]:
        info=z.getinfo(n); print('MEMBER',n,info.file_size)
        if info.file_size<2000000: print(z.read(n).decode('utf-8','replace')[:1200])
idx=json.loads(Path('/content/day10_resume_base_model/model.safetensors.index.json').read_text())
from collections import Counter
counts=Counter()
for key, shard in idx['weight_map'].items():
    if key.startswith('model.visual.merger.') or key.startswith('model.visual.deepstack_merger_list.'):
        group='projector'
    elif key.startswith('model.visual.'):
        group='vision'
    else:
        group='language'
    counts[group]+=1
print('PARAM_KEY_COUNTS',counts)
# Read parameter shapes cheaply from safetensors headers via safe_open
from safetensors import safe_open
from math import prod
base=Path('/content/day10_resume_base_model')
params=Counter()
for shard in sorted(set(idx['weight_map'].values())):
    with safe_open(str(base/shard),framework='pt',device='cpu') as f:
        for key in f.keys():
            if key.startswith('model.visual.merger.') or key.startswith('model.visual.deepstack_merger_list.'):
                group='projector'
            elif key.startswith('model.visual.'):
                group='vision'
            else:
                group='language'
            params[group]+=prod(f.get_slice(key).get_shape())
print('PARAMS',dict(params),'TOTAL',sum(params.values()))

E0_ARCHIVE_HITS []
PARAM_KEY_COUNTS Counter({'language': 398, 'vision': 291, 'projector': 24})
PARAMS {'language': 4022468096, 'vision': 306242560, 'projector': 109105152} TOTAL 4437815808


In [50]:
from pathlib import Path
import subprocess
pushrepo=Path('/content/pushrepo')
if not (pushrepo/'.git').exists():
    r=subprocess.run(['git','clone','https://github.com/5cuong/K4-Track-4-Day10-NguyenHoangCuong-2A202602473-VisionLanguageModel.git',str(pushrepo)],text=True,capture_output=True)
    print('CLONE',r.returncode,r.stdout[-1500:],r.stderr[-1500:])
    if r.returncode: raise RuntimeError('Git clone failed')
else: print('CLONE_EXISTS')
r=subprocess.run(['git','-C',str(pushrepo),'rev-parse','--is-shallow-repository'],capture_output=True,text=True); print('SHALLOW',r.stdout.strip())
if r.stdout.strip()=='true':
    r=subprocess.run(['git','-C',str(pushrepo),'fetch','--unshallow','origin'],capture_output=True,text=True); print('UNSHALLOW',r.returncode,r.stderr[-500:])
for args in [['rev-list','--count','HEAD'],['branch','--show-current'],['remote','get-url','origin'],['log','-8','--format=%h %s']]:
    r=subprocess.run(['git','-C',str(pushrepo),*args],capture_output=True,text=True); print('GIT',args[0],r.returncode,r.stdout.strip(),r.stderr.strip()[:250])
print('AUTH_HELPER',subprocess.run(['git','config','--global','credential.helper'],capture_output=True,text=True).stdout.strip())

CLONE 0  Cloning into '/content/pushrepo'...

SHALLOW false
GIT rev-list 0 2 
GIT branch 0 main 
GIT remote 0 https://github.com/5cuong/K4-Track-4-Day10-NguyenHoangCuong-2A202602473-VisionLanguageModel.git 
GIT log 0 91bc192 feat: implement Day 10 lab workflow
c2f3997 feat: Lab Day 10 — finetune Qwen3-VL-4B (LoRA) cho grounding hư hại mặt đường 
AUTH_HELPER 


In [51]:
from pathlib import Path
root=Path('/content/day10_repo')
for fn in ['lab_day10_solution.py','scripts/score.py']:
    lines=(root/fn).read_text().splitlines(); print('###',fn)
    keys=['1000','projector','float32','assistant','coordinate','clamp','normaliz','box']
    ix=[i for i,s in enumerate(lines) if any(k in s.lower() for k in keys)]
    for i in ix[:80]: print(f'{i+1}: {lines[i][:220]}')

### lab_day10_solution.py
35:     "crack, manhole. Output a JSON list of objects with label and bbox_2d "
36:     "coordinates normalized on a 0-1000 scale: "
37:     '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
41:     "train": {"images": 1422, "boxes": {"pothole": 1074, "crack": 1750, "manhole": 691}},
42:     "val": {"images": 276, "boxes": {"pothole": 66, "crack": 470, "manhole": 103}},
43:     "test": {"images": 311, "boxes": {"pothole": 121, "crack": 299, "manhole": 163}},
47: def yolo_to_box(label: str, yolo: list[float]) -> dict:
48:     """Convert normalized YOLO center-width-height to integer 0–1000 XYXY."""
50:     x1 = int(round(max(0.0, min(1000.0, (xc - width / 2) * 1000))))
51:     y1 = int(round(max(0.0, min(1000.0, (yc - height / 2) * 1000))))
52:     x2 = int(round(max(0.0, min(1000.0, (xc + width / 2) * 1000))))
53:     y2 = int(round(max(0.0, min(1000.0, (yc + height / 2) * 1000))))
55:         x2 = min(1000, x1 + 1)
57:         y2 = min(1000, y1 + 1)
58:    

In [52]:
import subprocess,json
root=Path('/content/day10_repo')
r=subprocess.run(['python','scripts/score.py','--pred','work/e3.jsonl','--split','test','--name','E3_WRONG_SCALE_DIAGNOSTIC','--out','/tmp/e3_wrong_scale.json','--coord','pixel'],cwd=root,text=True,capture_output=True)
print('RETURN',r.returncode); print(r.stdout); print(r.stderr[-1200:])
if r.returncode==0: print('WRONG_SCALE_F1',json.loads(Path('/tmp/e3_wrong_scale.json').read_text())['micro']['f1'])

RETURN 0
E3_WRONG_SCALE_DIAGNOSTIC [test] n=311 valid=1.000 P=0.000 R=0.000 F1=0.000 CI95=0.000-0.000 meanIoU=0.000 FP/img=1.62
   pothole  P=0.000 R=0.000 F1=0.000
   crack    P=0.000 R=0.000 F1=0.000
   manhole  P=0.000 R=0.000 F1=0.000


WRONG_SCALE_F1 0.0


In [53]:
from pathlib import Path
import json, subprocess
root=Path('/content/day10_repo')
manifest=json.loads((root/'work/runs/manifest.json').read_text())
for short,key in [('e1','E1_r16'),('e2','E2_r16'),('e3','E3_r16')]:
    p=root/f'report/evidence/{short}.json'
    d=json.loads(p.read_text())
    m=manifest[key]
    d['meta']={k:float(m[k]) if k in ('peak_vram_gb','train_minutes') else int(m[k]) for k in ('trainable_params','peak_vram_gb','train_minutes')}
    tmp=p.with_suffix('.json.tmp'); tmp.write_text(json.dumps(d,ensure_ascii=False,indent=2)+'\n',encoding='utf-8'); tmp.replace(p)
report = '''# Báo cáo Lab Day 10 — Qwen3-VL-4B grounding hư hại mặt đường

Họ tên: Nguyen Hoang Cuong
MSSV: 2A202602473
GPU huấn luyện: NVIDIA A100-SXM4-40GB
Dataset version: d10-21e4070301

## Bảng tổng hợp kết quả đo trên test

E0 không huấn luyện; VRAM và thời gian huấn luyện không áp dụng. Các thông số E1–E4 lấy từ manifest của các lần chạy thật. F1/CI/FP mỗi ảnh và F1 lớp lấy từ evidence của scorer trên cùng 311 ảnh test.

| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GB) | Phút train | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |
|---|---|---:|---:|---:|---:|---|---:|---:|---:|---:|
| E0 | Zero-shot | 0 | — | — | 0.0887 | [0.0630, 0.1156] | 0.897 | 0.1088 | 0.0321 | 0.1868 |
| E1 | LoRA LLM, rank 16 | 33,030,144 | 29.201 | 11.236 | 0.2629 | [0.2290, 0.3031] | 1.206 | 0.1827 | 0.1821 | 0.5271 |
| E2 | E1 + projector đầy đủ | 142,135,296 | 13.388 | 36.132 | 0.3482 | [0.3075, 0.3886] | 0.932 | 0.2525 | 0.2664 | 0.5836 |
| E3 | E2 + LoRA ViT, rank 16 | 148,426,752 | 10.132 | 45.520 | 0.4254 | [0.3832, 0.4680] | 0.875 | 0.2771 | 0.3480 | 0.6993 |
| E4 rank 4 | E3, rank 4 | 118,935,552 | 23.732 | 13.435 | 0.4078 | [0.3686, 0.4499] | 1.013 | 0.3180 | 0.3121 | 0.6758 |
| E4 rank 64 | E3, rank 64 | 266,391,552 | 28.316 | 13.946 | 0.4278 | [0.3884, 0.4688] | 0.961 | 0.2996 | 0.3645 | 0.6643 |

## Trả lời câu hỏi

### Câu 1 — Kiến trúc và tham số

Đếm từ các tensor trong safetensors của đúng base model dùng trong phiên chạy: language model 4,022,468,096 tham số; vision encoder 306,242,560; projector (merger và deepstack merger) 109,105,152; tổng 4,437,815,808. E1 cập nhật LoRA ở LLM, giữ phần nhìn đóng băng. E2 mở projector đầy đủ ngoài LoRA LLM. E3 thêm LoRA ở các khối attention/MLP của ViT. Số tham số thực sự trainable của từng setup được ghi trong bảng.

### Câu 2 — E0 zero-shot

E0 có parse_valid_rate 0.8328, F1 0.0887, với 40 TP, 279 FP và 543 FN. Pattern lỗi đo được rõ nhất là bỏ sót box (FN cao hơn TP nhiều) và chất lượng định vị/phân lớp crack yếu: crack có 8 TP, 191 FP, 291 FN, F1 0.0321. File raw work/e0.jsonl không còn trong runtime hoặc gói khôi phục nên không thể xác minh hai dạng lỗi văn bản phổ biến hay trích output thô; tôi không dựng ví dụ giả. Số liệu này là thống kê thật trong report/evidence/e0.json.

### Câu 3 — Thang tọa độ

Output huấn luyện/chấm chuẩn dùng bbox 0–1000. Khi dùng cùng dự đoán E3 nhưng cố ý yêu cầu scorer diễn giải chúng như pixel, F1 giảm từ 0.4254 xuống 0.0000 (311 ảnh, tất cả lớp F1 bằng 0). Scorer chuẩn hóa pixel bằng chiều rộng/cao ảnh rồi đưa về 0–1000; diễn giải tọa độ 0–1000 như pixel làm box co sai tỷ lệ nên IoU không đạt ngưỡng. Đây là phép kiểm tra đối chứng trên prediction E3, không phải lần huấn luyện mới.

### Câu 4 — Assistant-only loss

Loss chỉ tính trên câu trả lời assistant để mô hình học sinh JSON grounding đích, không học bắt chước prompt người dùng hoặc token ảnh. Nếu tính loss trên toàn chuỗi, phần lớn token prompt/ảnh không phải nội dung cần sinh sẽ chi phối gradient và làm giảm tín hiệu học cho nhãn, lớp và tọa độ. Pipeline kiểm tra target assistant kết thúc bằng im_end.

### Câu 5 — Projector và lượng tử hóa

Projector cần là tham số trainable đầy đủ; nếu bị lượng tử hóa 4-bit thì tensor không còn dạng trọng số floating-point thông thường để cập nhật đầy đủ bằng optimizer. Cấu hình chạy đã bỏ lượng tử hóa các module visual (manifest ghi vision_quantized=false); mã cast tham số trainable của projector sang float32 và lưu projector riêng cùng adapter. Khi inference, projector được nạp lại; nếu chỉ nạp LoRA thì E2/E3 sẽ mất phần cập nhật projector.

### Câu 6 — Chia split theo phiên quay

Chia ảnh ngẫu nhiên có thể đưa khung hình gần như trùng bối cảnh/đoạn đường từ cùng phiên quay vào cả train và test, làm test dễ hơn và F1 lạc quan. Dữ liệu được chia theo 18 session không giao nhau: train 1,422 ảnh (pothole 1,074; crack 1,750; manhole 691 box), validation 276 ảnh (66; 470; 103), test 311 ảnh (121; 299; 163).

### Câu 7 — So sánh E1/E2/E3

E1 đạt F1 0.2629; E2 tăng lên 0.3482 (+0.0852), F1 manhole tăng 0.5271 lên 0.5836. E3 đạt 0.4254 (+0.0773 so với E2) và có F1 cao nhất ở cả test tổng thể lẫn crack/manhole. Kết quả ủng hộ việc mở projector và thêm LoRA ViT trên split này; tuy nhiên CI95 của E2 [0.3075, 0.3886] và E3 [0.3832, 0.4680] còn chồng lấn, nên chưa đủ kết luận chênh lệch chắc chắn. Chi phí đo được lần lượt là E1 11.236 phút/29.201 GB, E2 36.132 phút/13.388 GB, E3 45.520 phút/10.132 GB. A100 40GB là GPU chạy các số liệu này.

### Câu 8 — Độ rộng khoảng tin cậy

Độ rộng CI95 là E0 0.0526, E1 0.0741, E2 0.0810, E3 0.0847. Với 311 ảnh, sai số lấy mẫu còn đáng kể; CI của E1 và E2 chỉ tách nhẹ, còn E2/E3 chồng lấn. Vì vậy nên xem thứ hạng F1 là bằng chứng định hướng trên test hiện có, không phải bảo đảm rằng E3 tốt hơn trong mọi mẫu dữ liệu.

### Câu 9 — Khác biệt giữa lớp

Ở setup tốt nhất E3, pothole là lớp yếu nhất (F1 0.2771; 32 TP, 78 FP, 89 FN), tiếp theo crack 0.3480 và manhole 0.6993. Test có 121 pothole box so với 299 crack và 163 manhole; ít ví dụ hơn có thể làm mô hình khó bao phủ biến thiên. Kích thước/độ tương phản và nhập nhằng ngữ nghĩa là giả thuyết cần kiểm tra thêm, không được xác nhận chỉ từ bảng tổng hợp. Hình overlay thực tế của sáu ảnh test nằm tại report/figures/best_validation_setup_test_examples.png.

### Câu 10 — E4 rank ablation

E4 chỉ thay rank LoRA quanh E3 rank 16: rank 4 cho F1 0.4078 [0.3686, 0.4499], rank 64 cho 0.4278 [0.3884, 0.4688], so với E3 0.4254 [0.3832, 0.4680]. Các CI chồng lấn; rank 64 chỉ hơn E3 0.0024 nhưng dùng 266.4M tham số trainable và 28.316 GB VRAM so với 148.4M và 10.132 GB của E3. Chưa có bằng chứng đáng tin rằng tăng rank cải thiện kết quả đủ bù chi phí.

## Nhìn lại

Điểm cần cải thiện là giữ lại work/e0.jsonl cùng với evidence để có thể phân loại lỗi định dạng và trích ví dụ output thô. E0 vẫn chấm được từ evidence đã lưu, nhưng không thể tái kiểm tra văn bản gốc. Trên test E3, pothole còn nhiều FN; nên thu thập thêm ví dụ/session độc lập và phân tích kích thước box trước khi kết luận nguyên nhân.
'''
(root/'REPORT.md').write_text(report,encoding='utf-8')
print('UPDATED_META',[(n,json.loads((root/f'report/evidence/{n}.json').read_text())['meta']) for n in ['e1','e2','e3']])
print('REPORT_BYTES',(root/'REPORT.md').stat().st_size,'PLACEHOLDER','<điền' in (root/'REPORT.md').read_text())
r=subprocess.run(['python','scripts/check_submission.py'],cwd=root,text=True,capture_output=True); print('CHECKER_RETURN',r.returncode); print(r.stdout); print(r.stderr[-1000:])
print('AUTHOR',subprocess.run(['git','-C','/content/pushrepo','log','-1','--format=%an%x09%ae'],text=True,capture_output=True).stdout.strip())
print('GIT_CONFIG',subprocess.run(['git','-C','/content/pushrepo','config','user.name'],text=True,capture_output=True).stdout.strip(),subprocess.run(['git','-C','/content/pushrepo','config','user.email'],text=True,capture_output=True).stdout.strip())

UPDATED_META [('e1', {'trainable_params': 33030144, 'peak_vram_gb': 29.2012, 'train_minutes': 11.2356}), ('e2', {'trainable_params': 142135296, 'peak_vram_gb': 13.387965679168701, 'train_minutes': 36.13225569638332}), ('e3', {'trainable_params': 148426752, 'peak_vram_gb': 10.132108688354492, 'train_minutes': 45.51973023421667})]
REPORT_BYTES 7131 PLACEHOLDER False
CHECKER_RETURN 0
[PASS] evidence e0
[PASS] evidence e1
[PASS] evidence e2
[PASS] evidence e3
[PASS] evidence e4 — 2 file
[PASS] e1 meta
[PASS] e2 meta
[PASS] e3 meta
[PASS] dataset_version nhất quán — ['d10-21e4070301']
[PASS] khớp data/splits/meta.json
[PASS] REPORT.md đã điền — còn chuỗi '<điền'
[WARN] git — không phải git repo

0 FAIL, 1 WARN


AUTHOR unknown	hoangcuong170825@gmail.com
GIT_CONFIG  


In [59]:
import os,subprocess
repo='/content/pushrepo'
def run(args, cwd=None):
 p=subprocess.run(args,cwd=cwd,text=True,capture_output=True,timeout=90,env={**os.environ,'GIT_TERMINAL_PROMPT':'0'})
 print('$',' '.join(args),'RETURN',p.returncode)
 if p.stdout: print(p.stdout[-3500:])
 if p.stderr: print('STDERR',p.stderr[-1800:])
 return p
print('GITHUB_AUTH_STATUS (no credentials are printed):')
a=run(['gh','auth','status'])
print('PUSH_REPO_EXISTS',os.path.isdir(repo), 'GIT_DIR',os.path.isdir(repo+'/.git'))
if os.path.isdir(repo+'/.git'):
 for args in [['git','status','--short'],['git','branch','--show-current'],['git','remote','-v'],['git','log','-8','--format=%H %s'],['git','rev-parse','HEAD'],['git','config','user.name'],['git','config','user.email']]: run(args,repo)
 print('SUBMISSION CHECKER:')
 run(['python','scripts/check_submission.py'],repo)


GITHUB_AUTH_STATUS (no credentials are printed):
$ gh auth status RETURN 1
STDERR You are not logged into any GitHub hosts. To log in, run: gh auth login

PUSH_REPO_EXISTS True GIT_DIR True
$ git status --short RETURN 0
$ git branch --show-current RETURN 0
main

$ git remote -v RETURN 0
origin	https://github.com/5cuong/K4-Track-4-Day10-NguyenHoangCuong-2A202602473-VisionLanguageModel.git (fetch)
origin	https://github.com/5cuong/K4-Track-4-Day10-NguyenHoangCuong-2A202602473-VisionLanguageModel.git (push)

$ git log -8 --format=%H %s RETURN 0
ce9310dd16384dae76f06df41e55febe8a2a5236 CP[REPORT-and-figures]
a01a6801af5d64277afbb37e19802eadf456c1e4 CP[E4-rank-ablation]
97aba81428b47e675bd155d2fe54fbf8d3a0862a CP[E0-E3-evidence]
91bc19237ed9c72995f1fc171be61a481d4b896b feat: implement Day 10 lab workflow
c2f39979458d5b10b1055c6ab47d6bd855b73151 feat: Lab Day 10 — finetune Qwen3-VL-4B (LoRA) cho grounding hư hại mặt đường

$ git rev-parse HEAD RETURN 0
ce9310dd16384dae76f06df41e55febe8a2a5236